# Customer Churn Forecasting - K-Nearest Neighbors (KNN)
## Module 5: Instance-Based Distance Classification

KNN classifier using Euclidean distance on standardized features to predict churn based on neighborhood voting.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 5.1 Load and Prepare Data

In [ ]:
# Load and prepare the dataset
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Preprocessing pipeline
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in df.columns)
df['ChargesPerService'] = np.where(df['TotalServices'] > 0, df['MonthlyCharges'] / df['TotalServices'], df['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

# Encode
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

# Split
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Scale features (critical for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Dataset prepared. Shape: {df.shape}')
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## 5.2 Find Optimal K Value

In [ ]:
# Test different K values to find optimal
k_range = range(1, 21)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    k_scores.append(scores.mean())
    if k % 5 == 0:
        print(f'K={k}: AUC={scores.mean():.4f}')

optimal_k = k_range[np.argmax(k_scores)]
print(f'\nOptimal K: {optimal_k} (AUC: {max(k_scores):.4f})')

# Plot K vs AUC
plt.figure(figsize=(10, 5))
plt.plot(k_range, k_scores, 'bo-', markersize=6, linewidth=2)
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal K={optimal_k}')
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('Cross-Validation ROC-AUC', fontsize=12)
plt.title('K Selection: Cross-Validation AUC Score', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.xticks(k_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('knn_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Train KNN with Optimal K

In [ ]:
# Train KNN with optimal K
knn_model = KNeighborsClassifier(
    n_neighbors=optimal_k,
    weights='distance',  # Weight closer neighbors more
    metric='euclidean',
    n_jobs=-1
)
knn_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

print('=' * 60)
print(f'K-NEAREST NEIGHBORS (K={optimal_k}) - RESULTS')
print('=' * 60)
print(f'\nAccuracy:  {accuracy_score(y_test, y_pred_knn)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_knn)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred_knn)*100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred_knn):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_knn):.4f}')

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_test, y_pred_knn, target_names=['Retained', 'Churned']))

## 5.4 Confusion Matrix

In [ ]:
# Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            annot_kws={'size': 16}, linewidths=1, linecolor='white')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title(f'KNN (K={optimal_k}) - Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('knn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_knn.ravel()
print(f'True Negatives:  {tn:,} | False Positives: {fp:,}')
print(f'False Negatives: {fn:,} | True Positives:  {tp:,}')

## 5.5 ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_knn)
auc_knn = roc_auc_score(y_test, y_prob_knn)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#27ae60', lw=2, label=f'KNN K={optimal_k} (AUC = {auc_knn:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='#27ae60')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - K-Nearest Neighbors', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('knn_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()